# 08. MCLP 최적배치 — 신규 AED 선정

## 이 노트북이 하는 일
공백(07)을 최대한 덮도록 신규 AED 위치를 고른다. **최대커버리지 입지문제(MCLP)** 를 greedy로 근사.

## 왜 이렇게 설계했나 (설계 이유)
- **왜 MCLP인가:** "정해진 개수의 시설로 최대 수요를 커버"하는 고전 입지최적화. AED 예산이 한정된 현실과 맞다.
- **왜 greedy인가:** 정확해는 정수계획법(ILP)이지만 solver 의존·느림. greedy(매번 가장 많이 새로 덮는 후보 선택)는 커버리지 문제에서 근사보장이 좋고 의존성 없이 빠르다.
- **왜 수요를 '위험>0 & 야간 미커버'로 한정하나:** 이미 야간 AED가 덮은 곳·위험 없는 곳은 신규 대상이 아님. 남은 위험만 목표로.
- **왜 가중치가 risk_norm인가:** 같은 커버라도 더 위험한 격자를 덮는 게 가치가 크다. MCLP 목적함수의 계수.
- **왜 후보지가 격자중심인가:** 실제 후보지(경로당 등) 좌표가 아직 없어, 일단 어느 격자든 설치 가능하다고 보고 최적 위치를 찾음. 실좌표 스냅은 09.
- **왜 커버리지 곡선을 보나:** 몇 개부터 효과가 체감되는지(한계효용) 확인 → 예산 대비 적정 개수 판단.

## 데이터 출처
- 공백 격자: 07(grid_gap_A4).

In [ ]:
%pip install scipy

In [ ]:
import os, warnings                       # 폴더·경고
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd            # 수치·표
import geopandas as gpd                     # 지리표
from scipy.spatial import cKDTree           # 후보-수요 커버 관계 빠르게
import folium, matplotlib                   # 지도·곡선
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["font.family"]="Malgun Gothic"; plt.rcParams["axes.unicode_minus"]=False  # 한글 폰트
CRS_WGS, CRS_M = 4326, 5186                 # 위경도 / 평면
R = 100                                     # 커버 반경(m) — 07 공백 진단과 동일(보수적)
N_NEW = 20                                  # 신규 AED 최대 개수
os.makedirs("outputs", exist_ok=True)

## 1. 데이터 — 공백 격자 (07 결과)

In [ ]:
grid = gpd.read_parquet("outputs/grid_gap_A4.parquet")                                # 07 공백 분석 격자
xy = np.c_[grid["cx"].values, grid["cy"].values]                                      # 전체 격자 좌표
demand_mask = (grid["risk_norm"]>0) & (~grid["covered_night"])                        # 수요 = 위험>0 이면서 야간 미커버
d_idx = np.where(demand_mask.values)[0]                                               # 수요 격자 인덱스
d_xy = xy[d_idx]; d_w = grid["risk_norm"].values[d_idx]                               # 수요 좌표 / 가중치(위험도)
print("전체 격자:", len(grid))
print("수요 격자(위험>0 & 야간 미커버):", len(d_idx), "| 위험가중 합:", round(d_w.sum(),1))
print("공백 격자(고위험):", int(grid["gap"].sum()))

## 2. greedy MCLP — 신규 N개 선정

In [ ]:
cand_xy = xy                                                                          # 후보지 = 모든 격자 중심
tree = cKDTree(d_xy)                                                                  # 수요 격자로 KD트리
cover = tree.query_ball_point(cand_xy, R)                                             # 후보별로 반경내 덮는 수요 격자 목록

covered = np.zeros(len(d_idx), dtype=bool)                                            # 이미 덮인 수요 표시
selected = []                                                                         # 선택된 후보(격자 index)
hist = []                                                                             # (개수, 누적 커버 가중치) 기록
for step in range(N_NEW):                                                             # N개까지 반복
    best_g, best_c = -1, -1                                                           # 이번에 최고 이득 후보
    for ci, dl in enumerate(cover):                                                   # 모든 후보 검토
        if not dl: continue
        g = d_w[[j for j in dl if not covered[j]]].sum()                              # 아직 안 덮인 수요의 위험가중 합 = 새 이득
        if g > best_g:                                                                # 가장 큰 이득 후보 갱신
            best_g, best_c = g, ci
    if best_c < 0 or best_g <= 0: break                                              # 더 덮을 게 없으면 종료
    for j in cover[best_c]: covered[j] = True                                         # 그 후보가 덮는 수요를 커버 처리
    selected.append(best_c)                                                           # 후보 확정
    hist.append((len(selected), covered_w:=d_w[covered].sum()))                       # 진행 기록

tot = d_w.sum()                                                                       # 전체 수요 위험가중
print(f"신규 {len(selected)}개 선정")
for n,cw in hist[:: max(1,len(hist)//10 or 1)]:                                        # 진행 곡선 요약 출력
    print(f"  {n:2d}개 → 위험가중 커버 {cw:.1f}/{tot:.1f} ({cw/tot*100:.0f}%)")
print(f"최종: {len(selected)}개로 미커버 위험의 {covered.sum()}/{len(d_idx)} 격자, 위험가중 {d_w[covered].sum()/tot*100:.0f}% 커버")

## 3. 신규 AED 후보 좌표 (WGS84)

In [ ]:
sel = grid.iloc[selected].copy()                                                      # 선택된 격자들
sel_w = sel.to_crs(CRS_WGS)                                                           # 위경도로
res = pd.DataFrame({                                                                  # 결과 표
    "rank": range(1, len(sel)+1),                                                     # 선정 순위(=중요도)
    "grid_id": sel["grid_id"].values,
    "dong": sel["dong"].values,
    "lon": sel_w.geometry.centroid.x.values,                                          # 제안 경도
    "lat": sel_w.geometry.centroid.y.values,                                          # 제안 위도
    "risk_norm": sel["risk_norm"].round(3).values,
})
res.to_csv("outputs/mclp_new_aed.csv", index=False, encoding="utf-8-sig")             # 신규 좌표 저장(09가 읽음)
print(res.to_string(index=False))
print("\n행정동별 신규 AED:"); print(sel["dong"].value_counts().to_string())

## 4. 커버리지 곡선 + 지도

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.5))                                               # 커버리지 곡선(개수 대비 커버율)
ns=[h[0] for h in hist]; cw=[h[1]/tot*100 for h in hist]
ax.plot(ns, cw, marker="o", color="#2c7fb8")
ax.set_xlabel("신규 AED 개수"); ax.set_ylabel("미커버 위험 커버율 (%)")
ax.set_title("MCLP 커버리지 곡선"); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("outputs/mclp_coverage_curve.png", dpi=120); plt.show()  # 곡선 저장

In [ ]:
newly = np.zeros(len(grid), dtype=bool)                                               # 신규로 커버된 수요 격자 표시
newly[d_idx[covered]] = True
gw = grid.to_crs(CRS_WGS)
def rcol(v):                                                                          # 위험도→색
    t=float(v); r=int(255*min(t+0.1,1)); g=int(200*(1-t)); b=int(60*(1-t)); return f"#{r:02x}{g:02x}{b:02x}"
m = folium.Map(location=[35.128,129.045], zoom_start=14, tiles="cartodbpositron")
fg=folium.FeatureGroup(name="위험도").add_to(m)                                        # 위험도 격자
for _,r in gw.iterrows():
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=lambda x,c=rcol(r["risk_norm"]):{"color":c,"weight":0.2,"fillColor":c,"fillOpacity":0.4}).add_to(fg)
fgn=folium.FeatureGroup(name="신규 커버 격자").add_to(m)                                # 신규로 덮인 격자(초록 반투명)
for i,(_,r) in enumerate(gw.iterrows()):
    if newly[i]:
        folium.GeoJson(r["geometry"].__geo_interface__,
            style_function=lambda x:{"color":"#1a9641","weight":1,"fillColor":"#1a9641","fillOpacity":0.15}).add_to(fgn)
for _,r in res.iterrows():                                                            # 신규 AED 위치(초록 +)
    folium.Marker([r["lat"],r["lon"]], tooltip=f'신규#{int(r["rank"])} {r["dong"]}',
        icon=folium.Icon(color="green", icon="plus", prefix="fa")).add_to(m)
folium.LayerControl().add_to(m)
m.save("outputs/mclp_map_A4.html")                                                     # MCLP 지도 저장
print("지도 저장: outputs/mclp_map_A4.html")
m